In [1]:
"""
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ney_gtr
!ls
"""

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n%cd /content/drive/MyDrive/ney_gtr\n!ls\n"

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import librosa
import numpy as np
import seaborn as sns
from constants import *
import IPython.display as ipd
import matplotlib.pyplot as plt
from utils import stitch_wave_chunks
from dataset import build_data_loaders

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
train_data_loader_abs, test_data_loader_abs = build_data_loaders("abs")
train_data_loader_angle, test_data_loader_angle = build_data_loaders("angle")

In [4]:
test_data_loader_abs.dataset.min_max_vals

{'gtr': {'min': -63.19031, 'max': 15.936092},
 'ney': {'min': -62.944256, 'max': 16.564707}}

In [5]:
test_data_loader_angle.dataset.min_max_vals

{'gtr': {'min': -3.1415887, 'max': 3.1415925},
 'ney': {'min': -3.1415884, 'max': 3.1415925}}

In [16]:
x, y, fp_x, fp_y = next(iter(test_data_loader_abs))
print(torch.min(x), torch.max(x))
print("Original shape:", x.shape)
NUM_TOTAL_FEATURES = int(x.shape[2] * x.shape[3])
print("NUM_TOTAL_FEATURES:", NUM_TOTAL_FEATURES)
print()

# encoder
print("Encoder:")
x = nn.Conv2d(1, 2, 3, stride=2)(x)
print("E1", x.size())
x = nn.Conv2d(2, 4, 3, stride=2)(x)
print("E2", x.size())
x = nn.BatchNorm2d(4)(x)
print("E3", x.size())
x = nn.Conv2d(4, 8, 3, stride=2)(x)
print("E4", x.size())
x = nn.Conv2d(8, 16, 3, stride=2)(x)
print("E5", x.size())
x = nn.BatchNorm2d(16)(x)
print("E6", x.size())
print()

# bottle
x = x.view(4, -1)
print("B1", x.size())
x = nn.Linear(1680, 8224)(x)
print("B2", x.size())
x = nn.Linear(8224, 32896)(x)
print("B2", x.size())
x = x.view(4, -1, 128, 257)
print("B3", x.size())
print()

# decoder
# print("Decoder:")
# x = nn.ConvTranspose2d(1, 32, 3, stride=1)(x)
# print("D1", x.size())
# x = nn.ConvTranspose2d(32, 16, 3, stride=1)(x)
# print("D2", x.size())
# x = nn.ConvTranspose2d(16, 4, 3, stride=1)(x)
# print("D3", x.size())
# x = nn.ConvTranspose2d(4, 1, 3, stride=1)(x)
# print("D4", x.size())

tensor(0.0449) tensor(0.9942)
Original shape: torch.Size([4, 1, 128, 257])
NUM_TOTAL_FEATURES: 32896

Encoder:
E1 torch.Size([4, 2, 63, 128])
E2 torch.Size([4, 4, 31, 63])
E3 torch.Size([4, 4, 31, 63])
E4 torch.Size([4, 8, 15, 31])
E5 torch.Size([4, 16, 7, 15])
E6 torch.Size([4, 16, 7, 15])

B1 torch.Size([4, 1680])
B2 torch.Size([4, 8224])
B2 torch.Size([4, 32896])
B3 torch.Size([4, 1, 128, 257])



In [17]:
class Gtr_2_Ney_AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.NUM_TOTAL_FEATURES = 32896
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 2, 3, stride=2),
            nn.ReLU(),
            nn.Conv2d(2, 4, 3, stride=2),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Conv2d(4, 8, 3, stride=2),
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        # self.decoder = nn.Sequential(
        #     nn.ConvTranspose2d(1, 32, 3, stride=1),
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(32, 16, 3, stride=1),
        #     nn.BatchNorm2d(16),
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(16, 4, 3, stride=1),
        #     nn.ReLU(),
        #     nn.ConvTranspose2d(4, 1, 3, stride=1),
        #     nn.BatchNorm2d(1),
        #     nn.Sigmoid()
        # )

        self.fc = nn.Sequential(
            nn.Linear(1680, 8224),
            nn.Sigmoid(),
            nn.Linear(8224, 32896),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(4, -1)
        x = self.fc(x)
        x = x.view(4, -1, 128, 257)

        return x

In [18]:
def train_model(part="abs", lr=0.001, num_epochs=10):
    train_data_loader, test_data_loader = build_data_loaders(part)
    # model = Gtr_2_Ney_AutoEncoder().to(device)
    model = Gtr_2_Ney_AutoEncoder()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}

    for epoch in range(num_epochs):
        running_loss = 0.0
        model.train(True)
        num_train_batches = 0
        for gtr_features, ney_features, _, _ in train_data_loader:
            # gtr_features = gtr_features.to(device)
            # ney_features = ney_features.to(device)
            y_hat = model(gtr_features)
            loss = criterion(y_hat, ney_features)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            num_train_batches += 1

        model.eval()
        val_loss = 0.0
        num_val_batches = 0
        with torch.no_grad():
            for gtr_features, ney_features, _, _ in test_data_loader:
                if len(gtr_features) != 4:
                    continue
                # gtr_features = gtr_features.to(device)
                # ney_features = ney_features.to(device)
                y_hat = model(gtr_features)
                loss = criterion(y_hat, ney_features)
                val_loss += loss.item()
                num_val_batches += 1

        train_loss = running_loss / num_train_batches
        val_loss = val_loss / num_val_batches

        print(
            f"E: {epoch + 1:03d}/{num_epochs}\t T: {train_loss:.6f}\t V: {val_loss:.6f}")

        history["train"].append(train_loss)
        history["val"].append(val_loss)

    return model, history

In [19]:
# Epoch: 060/60	Loss: 0.000470
model_abs, history_abs = train_model(
    part="abs", 
    lr=0.0002, 
    num_epochs=60)

E: 001/60	 T: 0.005300	 V: 0.021821
E: 002/60	 T: 0.002462	 V: 0.003625


KeyboardInterrupt: 

In [23]:
def plot_loss(loss_graph, title, start=0):
    plt.figure(figsize=(8, 4))
    plt.title(title)
    epochs = np.arange(1 + start, len(loss_graph) + 1)
    plt.plot(epochs, loss_graph[start:])
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.show()

In [ ]:
plot_loss(loss_graph_abs, "Abs Loss", start=2)

In [ ]:
model_angle, test_data_loader_angle, loss_graph_angle = train_model(
    part="angle", 
    lr=0.001, 
    num_epochs=60)

In [ ]:
plot_loss(loss_graph_angle, "Angle Loss", start=2)

In [32]:
def predict(model, data_loader, part):
    mm_vals = data_loader.dataset.min_max_vals["ney"]
    x, y, x_paths, y_paths = next(iter(data_loader))
    print(x_paths)
    print(y_paths)
    print("-" * 20)
    predicted_chunks = None
    with torch.no_grad():
        predicted_chunks = model(x).numpy()
    
    predicted_chunks = np.squeeze(predicted_chunks, axis=1)
    predicted_chunks = (predicted_chunks * (mm_vals["max"] - mm_vals["min"])) + mm_vals["min"]
    target_chunks = np.squeeze(y.numpy(), axis=1)
    target_chunks = (target_chunks * (mm_vals["max"] - mm_vals["min"])) + mm_vals["min"]
    if part == "abs":
        predicted_chunks = librosa.db_to_power(predicted_chunks)
        target_chunks = librosa.db_to_power(target_chunks)

    return predicted_chunks, target_chunks

In [ ]:
predicted_chunks_abs, target_chunks_abs = predict(
    model_abs, test_data_loader_abs, "abs")
predicted_chunks_angle, target_chunks_angle = predict(
    model_angle, test_data_loader_angle, "angle")

In [55]:
def plot_heatmaps(prediction, target):
    sns.set_theme(rc={"figure.figsize": (14, 5)})
    _, (ax1, ax2) = plt.subplots(1,2)
    ax1 = sns.heatmap(librosa.power_to_db(prediction), ax=ax1)
    ax1.set_title("Predicted")
    ax1.invert_yaxis()

    ax2 = sns.heatmap(librosa.power_to_db(target), ax=ax2)
    ax2.set_title("Actual")
    ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_heatmaps(predicted_chunks_abs[0], target_chunks_abs[0])

In [ ]:
plot_heatmaps(predicted_chunks_angle[0], target_chunks_angle[0])

In [46]:
def make_wav(abs_chunks, angle_chunks):
    wave_chunks = []
    for chunk_abs, chunk_ang in zip(abs_chunks, angle_chunks):
        chunk = chunk_abs * (np.cos(chunk_ang) + 1j*np.sin(chunk_ang))
        wave_chunk = librosa.istft(chunk, n_fft=N_FFT, hop_length=HOP)
        wave_chunks.append(wave_chunk)

    stitched_wave = stitch_wave_chunks(wave_chunks)
    return stitched_wave

In [ ]:
wave_prediction = make_wav(predicted_chunks_abs[:3], target_chunks_angle[:3])
wave_target = make_wav(target_chunks_abs[:3], target_chunks_angle[:3])
print(len(wave_prediction), len(wave_target))

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 6))
fig.suptitle("Target & Predicted Waves")
axs[0].set_title("Target")
axs[0].plot(wave_target)
axs[1].set_title("Prediction")
axs[1].plot(wave_prediction)
fig.tight_layout()
plt.show()

In [ ]:
ipd.Audio(wave_target, rate=SR)

In [ ]:
ipd.Audio(wave_prediction, rate=SR)